# 📜 Document Denoising — Denoising Autoencoder (U-Net)
### Kaggle: Denoising Dirty Documents

**Features:**
- ✅ Resume from any epoch (checkpoint every epoch with timestamp)
- ✅ PSNR + SSIM metrics tracked
- ✅ Google Drive sync for checkpoints
- ✅ ~90–120 min training on Colab GPU (T4)
- ✅ Best model saved for local Flask deployment

---
**Steps:** `Setup` → `Dataset` → `Train` → `Evaluate` → `Download Model`

## Step 1 — GPU Check & Install Packages

In [ ]:
import subprocess, sys

# Check GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected!')
    print(result.stdout.split('\n')[8])
else:
    print('⚠️  No GPU found. Go to: Runtime → Change runtime type → GPU (T4)')

# Install packages
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'kaggle', 'scikit-image', 'tqdm'], check=True)
print('✅ Packages installed')

## Step 2 — Mount Google Drive (for persistent checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/DocumentDenoising'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/outputs', exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_DIR}')

## Step 3 — Kaggle Dataset Download

> **One-time setup:** Upload your `kaggle.json` API key below.  
> Get it from: kaggle.com → Account → Create API Token

In [ ]:
from google.colab import files
import os

# Upload kaggle.json
print('Upload your kaggle.json file:')
uploaded = files.upload()

# Configure kaggle
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✅ Kaggle API configured')

In [ ]:
import subprocess, os

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(f'{DATA_DIR}/train'):
    print('Downloading dataset...')
    subprocess.run(['kaggle', 'competitions', 'download',
                    '-c', 'denoising-dirty-documents',
                    '-p', DATA_DIR], check=True)
    subprocess.run(['unzip', '-q', f'{DATA_DIR}/denoising-dirty-documents.zip',
                    '-d', DATA_DIR], check=True)
    print('✅ Dataset downloaded and extracted')
else:
    print('✅ Dataset already present, skipping download')

# Show what we have
for split in ['train', 'train_cleaned', 'test']:
    path = os.path.join(DATA_DIR, split)
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f'  {split}/  →  {count} images')

## Step 4 — Model & Training Code

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ─── U-Net Blocks ─────────────────────────────────────────────────────────────

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class DenoisingUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.enc = nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)
        ch = in_ch
        for f in features:
            self.enc.append(ConvBlock(ch, f))
            ch = f
        self.bottleneck = ConvBlock(features[-1], features[-1]*2)
        self.dec = nn.ModuleList()
        ch = features[-1]*2
        for f in reversed(features):
            self.dec.append(UpBlock(ch, f, f))
            ch = f
        self.out_conv = nn.Sequential(
            nn.Conv2d(features[0], out_ch, 1), nn.Sigmoid()
        )
    def forward(self, x):
        skips = []
        for enc in self.enc:
            x = enc(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        for i, dec in enumerate(self.dec):
            x = dec(x, skips[-(i+1)])
        return self.out_conv(x)
    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = DenoisingUNet().to(DEVICE)
print(f'Device      : {DEVICE}')
print(f'Parameters  : {model.count_params():,}')
# Quick shape test
with torch.no_grad():
    dummy = torch.randn(2, 1, 128, 128).to(DEVICE)
    out = model(dummy)
print(f'Output shape: {out.shape}  range [{out.min():.2f}, {out.max():.2f}]')

In [ ]:
import os, random, math
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class DirtyDocDataset(Dataset):
    def __init__(self, noisy_dir, clean_dir, patch_size=128,
                 patches_per_img=40, mode='train', val_split=0.1, seed=42):
        self.noisy_dir, self.clean_dir = noisy_dir, clean_dir
        self.patch_size = patch_size
        self.patches_per_img = patches_per_img
        self.mode = mode
        self.augment = (mode == 'train')

        all_noisy = sorted([f for f in os.listdir(noisy_dir)
                            if f.lower().endswith(('.png','.jpg','.tif'))])
        all_clean = sorted([f for f in os.listdir(clean_dir)
                            if f.lower().endswith(('.png','.jpg','.tif'))])

        rng = random.Random(seed)
        idx = list(range(len(all_noisy)))
        rng.shuffle(idx)
        n_val = max(1, int(len(idx)*val_split))
        idx = idx[n_val:] if mode=='train' else idx[:n_val]

        self.noisy_files = [all_noisy[i] for i in idx]
        self.clean_files = [all_clean[i] for i in idx]

    def __len__(self):
        return len(self.noisy_files) * (self.patches_per_img if self.mode=='train' else 1)

    def _load(self, path):
        return np.array(Image.open(path).convert('L'), dtype=np.float32) / 255.0

    def __getitem__(self, idx):
        img_idx = idx // self.patches_per_img if self.mode=='train' else idx
        n = self._load(os.path.join(self.noisy_dir, self.noisy_files[img_idx]))
        c = self._load(os.path.join(self.clean_dir, self.clean_files[img_idx]))

        if self.mode == 'train':
            ps = self.patch_size
            h, w = n.shape
            if h < ps: n = np.pad(n,((0,ps-h),(0,0)),'reflect'); c = np.pad(c,((0,ps-h),(0,0)),'reflect'); h=ps
            if w < ps: n = np.pad(n,((0,0),(0,ps-w)),'reflect'); c = np.pad(c,((0,0),(0,ps-w)),'reflect'); w=ps
            t = random.randint(0, h-ps); l = random.randint(0, w-ps)
            n, c = n[t:t+ps, l:l+ps], c[t:t+ps, l:l+ps]
            if self.augment:
                if random.random()>0.5: n=np.fliplr(n).copy(); c=np.fliplr(c).copy()
                if random.random()>0.5: n=np.flipud(n).copy(); c=np.flipud(c).copy()
                k=random.randint(0,3); n=np.rot90(n,k).copy(); c=np.rot90(c,k).copy()

        nt = torch.from_numpy(n).unsqueeze(0)
        ct = torch.from_numpy(c).unsqueeze(0)
        return nt, ct

NOISY_DIR = f'{DATA_DIR}/train'
CLEAN_DIR = f'{DATA_DIR}/train_cleaned'

train_ds = DirtyDocDataset(NOISY_DIR, CLEAN_DIR, patch_size=128, patches_per_img=40, mode='train')
val_ds   = DirtyDocDataset(NOISY_DIR, CLEAN_DIR, patch_size=128, patches_per_img=1,  mode='val')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=1,  shuffle=False, num_workers=1, pin_memory=True)

print(f'Train : {len(train_ds):,} patches  ({len(train_loader)} batches)')
print(f'Val   : {len(val_ds)} full images')

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math

# ─── PSNR & SSIM ──────────────────────────────────────────────────────────────

def psnr(pred, target):
    mse = F.mse_loss(pred, target).item()
    return float('inf') if mse == 0 else 20*math.log10(1.0) - 10*math.log10(mse)

def ssim(pred, target, C1=0.01**2, C2=0.03**2):
    kernel = torch.ones(1,1,11,11, device=pred.device)/(11*11)
    def mu(x): return F.conv2d(x, kernel, padding=5)
    mx,my = mu(pred), mu(target)
    sx = mu(pred**2)-mx**2; sy = mu(target**2)-my**2; sxy = mu(pred*target)-mx*my
    num = (2*mx*my+C1)*(2*sxy+C2)
    den = (mx**2+my**2+C1)*(sx+sy+C2)
    return (num/den).mean().item()

# ─── Combined Loss ────────────────────────────────────────────────────────────

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha
    def forward(self, pred, target):
        mse = F.mse_loss(pred, target)
        # SSIM loss (differentiable)
        C1,C2 = 0.01**2, 0.03**2
        k = torch.ones(1,1,11,11,device=pred.device)/(11*11)
        def mu(x): return F.conv2d(x,k,padding=5)
        mx,my = mu(pred),mu(target)
        sx=mu(pred**2)-mx**2; sy=mu(target**2)-my**2; sxy=mu(pred*target)-mx*my
        ssim_val = ((2*mx*my+C1)*(2*sxy+C2)/((mx**2+my**2+C1)*(sx+sy+C2))).mean()
        return self.alpha*mse + (1-self.alpha)*(1-ssim_val)

# Quick test
loss_fn = CombinedLoss(alpha=0.5)
a = torch.rand(2,1,64,64); b = torch.rand(2,1,64,64)
print(f'PSNR={psnr(a,b):.2f} dB | SSIM={ssim(a,b):.4f} | Loss={loss_fn(a,b).item():.4f}')
print('✅ Loss and metrics OK')

## Step 5 — Training (with auto-resume & Drive backup)

In [ ]:
# ─── Training Config ──────────────────────────────────────────────────────────
CONFIG = {
    'num_epochs'    : 100,
    'learning_rate' : 1e-3,
    'weight_decay'  : 1e-4,
    'grad_clip'     : 1.0,
    'local_ckpt_dir': '/content/checkpoints',
    'drive_ckpt_dir': f'{DRIVE_DIR}/checkpoints',
    'output_dir'    : f'{DRIVE_DIR}/outputs',
    'resume'        : True,   # Set False to restart from scratch
}

os.makedirs(CONFIG['local_ckpt_dir'], exist_ok=True)
os.makedirs(CONFIG['drive_ckpt_dir'], exist_ok=True)
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print('Config:', CONFIG)

In [ ]:
import shutil, time, json

def save_checkpoint(epoch, model, optimizer, scheduler, history, best_psnr):
    state = {
        'epoch'    : epoch,
        'model_state'    : model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'history'  : history,
        'best_psnr': best_psnr,
        'saved_at' : time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    local_path = f"{CONFIG['local_ckpt_dir']}/latest.pth"
    epoch_path = f"{CONFIG['local_ckpt_dir']}/epoch_{epoch:04d}.pth"
    torch.save(state, local_path)
    torch.save(state, epoch_path)
    # Sync to Drive
    shutil.copy(local_path, f"{CONFIG['drive_ckpt_dir']}/latest.pth")
    shutil.copy(epoch_path, f"{CONFIG['drive_ckpt_dir']}/epoch_{epoch:04d}.pth")

def load_checkpoint(model, optimizer, scheduler):
    # Try local first, then Drive
    for path in [f"{CONFIG['local_ckpt_dir']}/latest.pth",
                 f"{CONFIG['drive_ckpt_dir']}/latest.pth"]:
        if os.path.exists(path):
            print(f'Loading checkpoint: {path}')
            ckpt = torch.load(path, map_location=DEVICE)
            model.load_state_dict(ckpt['model_state'])
            optimizer.load_state_dict(ckpt['optimizer_state'])
            scheduler.load_state_dict(ckpt['scheduler_state'])
            print(f"  Resumed epoch {ckpt['epoch']} | Best PSNR {ckpt['best_psnr']:.2f} dB | Saved {ckpt['saved_at']}")
            return ckpt['epoch'], ckpt['history'], ckpt['best_psnr']
    print('No checkpoint found — starting fresh.')
    return 0, {k:[] for k in ['train_loss','val_loss','train_psnr','val_psnr','train_ssim','val_ssim','epoch_times','lr']}, 0.0

print('✅ Checkpoint functions defined')

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import time

# ─── Setup ────────────────────────────────────────────────────────────────────
optimizer = Adam(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['num_epochs'], eta_min=1e-6)
loss_fn   = CombinedLoss(alpha=0.5)

history = {k:[] for k in ['train_loss','val_loss','train_psnr','val_psnr','train_ssim','val_ssim','epoch_times','lr']}
start_epoch, best_psnr = 0, 0.0

if CONFIG['resume']:
    start_epoch, history, best_psnr = load_checkpoint(model, optimizer, scheduler)

print(f'\nTraining: epoch {start_epoch+1} → {CONFIG["num_epochs"]}')
print(f'Device: {DEVICE} | Params: {model.count_params():,}\n')

# ─── Training Loop ────────────────────────────────────────────────────────────
total_start = time.time()

for epoch in range(start_epoch+1, CONFIG['num_epochs']+1):
    ep_start = time.time()

    # ── Train ──
    model.train()
    t_loss, t_psnr, t_ssim, n = 0.,0.,0.,0
    for noisy, clean in train_loader:
        noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        pred = model(noisy)
        loss = loss_fn(pred, clean)
        loss.backward()
        if CONFIG['grad_clip']>0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        optimizer.step()
        with torch.no_grad():
            t_loss+=loss.item(); t_psnr+=psnr(pred,clean); t_ssim+=ssim(pred,clean); n+=1

    # ── Validate ──
    model.eval()
    v_loss, v_psnr, v_ssim, m = 0.,0.,0.,0
    with torch.no_grad():
        for noisy, clean in val_loader:
            noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
            pred = model(noisy)
            v_loss+=loss_fn(pred,clean).item(); v_psnr+=psnr(pred,clean); v_ssim+=ssim(pred,clean); m+=1

    scheduler.step()
    ep_time = time.time() - ep_start
    current_lr = optimizer.param_groups[0]['lr']

    # Averages
    metrics = {
        'train_loss': t_loss/n, 'val_loss': v_loss/m,
        'train_psnr': t_psnr/n, 'val_psnr': v_psnr/m,
        'train_ssim': t_ssim/n, 'val_ssim': v_ssim/m,
        'epoch_times': ep_time, 'lr': current_lr,
    }
    for k,v in metrics.items(): history[k].append(v)

    is_best = metrics['val_psnr'] > best_psnr
    if is_best:
        best_psnr = metrics['val_psnr']
        torch.save(model.state_dict(), f"{CONFIG['output_dir']}/best_model.pth")

    # Save checkpoint (every epoch)
    save_checkpoint(epoch, model, optimizer, scheduler, history, best_psnr)

    # ETA
    avg_time = sum(history['epoch_times']) / len(history['epoch_times'])
    remaining = (CONFIG['num_epochs'] - epoch) * avg_time
    eta_m, eta_s = divmod(int(remaining), 60)
    eta_h, eta_m = divmod(eta_m, 60)
    eta_str = f'{eta_h}h{eta_m}m{eta_s}s' if eta_h else f'{eta_m}m{eta_s}s'

    star = '★' if is_best else ' '
    total_min = (time.time()-total_start)/60
    print(f'Ep {epoch:3d}/{CONFIG["num_epochs"]} {star}'
          f' | Loss {metrics["train_loss"]:.4f}/{metrics["val_loss"]:.4f}'
          f' | PSNR {metrics["train_psnr"]:.2f}/{metrics["val_psnr"]:.2f} dB'
          f' | SSIM {metrics["val_ssim"]:.4f}'
          f' | {ep_time:.1f}s | ETA {eta_str} | Total {total_min:.1f}min')

print(f'\n✅ Training complete! Best Val PSNR: {best_psnr:.2f} dB')
print(f'Model saved to: {CONFIG["output_dir"]}/best_model.pth')

## Step 6 — Training Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

ep = range(1, len(history['train_loss'])+1)

fig = plt.figure(figsize=(16, 10), facecolor='#1a1208')
fig.suptitle('Training History — Document Denoising U-Net',
             fontsize=16, color='#f5edd6', fontweight='bold', y=1.01)

gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

style = {'facecolor':'#1a1208', 'linewidth': 0}
colors = {'train':'#c4962a','val':'#e08060','grid':'rgba(196,150,42,0.15)'}

def styled_ax(ax, title, ylabel):
    ax.set_facecolor('#0f0a04')
    ax.tick_params(colors='#8a7a5a', labelsize=8)
    ax.set_title(title, color='#f5edd6', fontsize=11, pad=8)
    ax.set_ylabel(ylabel, color='#8a7a5a', fontsize=9)
    ax.set_xlabel('Epoch', color='#8a7a5a', fontsize=9)
    ax.grid(True, color='#3a2a10', alpha=0.5, linestyle='--', linewidth=0.5)
    for spine in ax.spines.values(): spine.set_color('#3a2a10')
    ax.legend(fontsize=8, facecolor='#1a1208', labelcolor='#f5edd6', edgecolor='#3a2a10')

# Loss
ax1 = fig.add_subplot(gs[0,0])
ax1.plot(ep, history['train_loss'], color='#c4962a', label='Train', linewidth=1.5)
ax1.plot(ep, history['val_loss'],   color='#e08060', label='Val',   linewidth=1.5)
styled_ax(ax1, 'Combined Loss (MSE + SSIM)', 'Loss')

# PSNR
ax2 = fig.add_subplot(gs[0,1])
ax2.plot(ep, history['train_psnr'], color='#c4962a', label='Train', linewidth=1.5)
ax2.plot(ep, history['val_psnr'],   color='#e08060', label='Val',   linewidth=1.5)
ax2.axhline(30, color='#6ab04c', linestyle='--', linewidth=1, alpha=0.7, label='Target (30 dB)')
styled_ax(ax2, 'PSNR (dB) ↑', 'PSNR (dB)')

# SSIM
ax3 = fig.add_subplot(gs[1,0])
ax3.plot(ep, history['train_ssim'], color='#c4962a', label='Train', linewidth=1.5)
ax3.plot(ep, history['val_ssim'],   color='#e08060', label='Val',   linewidth=1.5)
ax3.set_ylim(0, 1)
styled_ax(ax3, 'SSIM ↑', 'SSIM')

# Epoch Time
ax4 = fig.add_subplot(gs[1,1])
ax4.bar(ep, history['epoch_times'], color='#c4962a', alpha=0.7, width=0.8)
ax4.axhline(sum(history['epoch_times'])/len(history['epoch_times']),
            color='#e08060', linestyle='--', linewidth=1, label='Mean')
styled_ax(ax4, 'Epoch Time (s)', 'Seconds')

plt.tight_layout()
plot_path = f"{CONFIG['output_dir']}/training_curves.png"
plt.savefig(plot_path, dpi=120, bbox_inches='tight', facecolor='#1a1208')
plt.show()
print(f'Plot saved: {plot_path}')
print(f'\nFinal Results:')
print(f'  Best Val PSNR : {best_psnr:.2f} dB')
print(f'  Final Val SSIM: {history["val_ssim"][-1]:.4f}')
print(f'  Total Training: {sum(history["epoch_times"])/60:.1f} min')

## Step 7 — Visual Evaluation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Load best model
best_path = f"{CONFIG['output_dir']}/best_model.pth"
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

def quick_denoise(img_np, model, device):
    """Denoise a small image quickly (Colab display)."""
    # Resize to ≤512 for speed
    h, w = img_np.shape
    if max(h,w) > 512:
        scale = 512/max(h,w)
        pil = Image.fromarray((img_np*255).astype(np.uint8))
        pil = pil.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
        img_np = np.array(pil)/255.0
    t = torch.from_numpy(img_np.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(t)
    return out.squeeze().cpu().numpy()

# Show 4 examples
val_noisy = sorted(os.listdir(NOISY_DIR))[:4]
val_clean = sorted(os.listdir(CLEAN_DIR))[:4]

fig, axes = plt.subplots(4, 3, figsize=(12, 14), facecolor='#1a1208')
fig.suptitle('Denoising Results: Noisy → Denoised → Ground Truth',
             fontsize=13, color='#f5edd6', fontweight='bold')

for row, (nf, cf) in enumerate(zip(val_noisy, val_clean)):
    noisy_np = np.array(Image.open(os.path.join(NOISY_DIR, nf)).convert('L'))/255.0
    clean_np = np.array(Image.open(os.path.join(CLEAN_DIR, cf)).convert('L'))/255.0
    denoised = quick_denoise(noisy_np, model, DEVICE)

    # Compute metrics on resized
    h, w = denoised.shape
    clean_resized = np.array(Image.fromarray((clean_np*255).astype(np.uint8)).resize((w,h), Image.LANCZOS))/255.0
    pred_t = torch.from_numpy(denoised).unsqueeze(0).unsqueeze(0)
    gt_t   = torch.from_numpy(clean_resized.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    p_val  = psnr(pred_t, gt_t)
    s_val  = ssim(pred_t, gt_t)

    for col, (img, title) in enumerate([
        (noisy_np,   f'Noisy\n{nf}'),
        (denoised,   f'Denoised\nPSNR={p_val:.1f}dB  SSIM={s_val:.3f}'),
        (clean_resized, 'Ground Truth'),
    ]):
        ax = axes[row][col]
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.set_title(title, fontsize=8, color='#f5edd6', pad=4)
        ax.axis('off')

plt.tight_layout()
eval_path = f"{CONFIG['output_dir']}/visual_evaluation.png"
plt.savefig(eval_path, dpi=100, bbox_inches='tight', facecolor='#1a1208')
plt.show()
print(f'Evaluation grid saved: {eval_path}')

## Step 8 — Download Model for Local Deployment

In [ ]:
from google.colab import files
import shutil

# Copy best model locally for download
local_model = '/content/best_model.pth'
shutil.copy(f"{CONFIG['output_dir']}/best_model.pth", local_model)

print('Downloading best_model.pth ...')
print('Place it in the  outputs/  folder of the project on your local machine.')
files.download(local_model)

In [ ]:
# Also download training curves
files.download(f"{CONFIG['output_dir']}/training_curves.png")
files.download(f"{CONFIG['output_dir']}/visual_evaluation.png")
print('Done!')

## Step 9 — (Optional) Resume After Colab Disconnect

If Colab disconnects mid-training, just:
1. Re-run **Steps 1–5** (setup, drive mount, dataset check).
2. Make sure `CONFIG['resume'] = True` *(it's on by default)*.
3. Run the training cell — it will auto-load the latest checkpoint from Drive and continue from the last saved epoch.

All per-epoch `.pth` files are saved to `MyDrive/DocumentDenoising/checkpoints/`.